# Image Super-Resolution with Real-ESRGAN

This notebook upscales a low-resolution image 4x using the pretrained Real-ESRGAN model (`RealESRGAN_x4plus`). It creates a small low-resolution test image, runs Real-ESRGAN on it, and compares the result side by side with simple bicubic resizing.

**How to run:** Open in Google Colab with *Runtime → Change runtime type → GPU* and run all cells. The model code and pretrained weights are downloaded from the official Real-ESRGAN GitHub repository.

_Rewritten from my original Colab notebook: `Untitled0.ipynb`._

## 1. Install Real-ESRGAN

Newer torchvision versions removed `torchvision.transforms.functional_tensor`, which the `basicsr` dependency still imports, so we patch that one import after installing.

In [ ]:
!pip install -q basicsr facexlib gfpgan realesrgan
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' $(python -c "import site; print(site.getsitepackages()[0])")/basicsr/data/degradations.py
!git clone -q https://github.com/xinntao/Real-ESRGAN.git
!wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P Real-ESRGAN/weights

## 2. Create a low-resolution input image

We take a sample photo from scikit-image and shrink it to 1/4 size. To use your own photo, upload it and save it as `inputs/input.png`.

In [ ]:
import os
from PIL import Image
from skimage import data

os.makedirs("inputs", exist_ok=True)
original = Image.fromarray(data.astronaut())
low_res = original.resize((original.width // 4, original.height // 4), Image.BICUBIC)
low_res.save("inputs/input.png")
print("Low-res size:", low_res.size)

## 3. Run Real-ESRGAN (4x upscale)

In [ ]:
!python Real-ESRGAN/inference_realesrgan.py -n RealESRGAN_x4plus --model_path Real-ESRGAN/weights/RealESRGAN_x4plus.pth -i inputs/input.png -o outputs --outscale 4

## 4. Compare bicubic vs Real-ESRGAN

In [ ]:
import matplotlib.pyplot as plt

upscaled = Image.open("outputs/input_out.png")
bicubic = low_res.resize(upscaled.size, Image.BICUBIC)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [low_res, bicubic, upscaled],
                          ["Low-res input", "Bicubic 4x", "Real-ESRGAN 4x"]):
    ax.imshow(img); ax.set_title(title); ax.axis("off")
plt.show()